# Remote model timing on the validation split


## Setup

In [1]:
import json
from collections import OrderedDict
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
import time

import modules.llms as llm_parsers
from modules.utils import compare_preds, format_time

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

In [2]:
# Only used for the local embedding model in the example-matching strategy;
# the address-parsing model itself runs on the remote inference server.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

Embedding device: cuda


## Load dataset

Unlike `cross_val_evaluation.ipynb`, we keep the original train/val split
instead of merging and re-splitting into cross-validation folds: the train
split is the few-shot example pool, the val split is what gets evaluated.

In [3]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

train_data = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
val_data = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)

print(f"train: {len(train_data)} addresses, val: {len(val_data)} addresses")
display(val_data.sample(5))

train: 771 addresses, val: 152 addresses


,card_id,field,FullAddress,UnitNumber,HouseNumber,StreetName,Neighborhood,City,District,Region,State,Country,PostalCode
66,val_36,ApplicantCurrentAddress,Ittlingen Krs. Sinsheim,NaN,NaN,NaN,NaN,Ittlingen,Sinsheim,NaN,NaN,NaN,NaN
2,val_1,VictimCurrentAddress,"Nürnberg, Nibelungenstrasse 8",NaN,8,Nibelungenstrasse,NaN,Nürnberg,NaN,NaN,NaN,NaN,NaN
51,val_28,ApplicantCurrentAddress,Tirschenreuth Maximiliansplatz 1/Kreis Oberpfalz,NaN,1,Maximiliansplatz,NaN,Tirschenreuth,Oberpfalz,NaN,NaN,NaN,NaN
109,val_56,VictimBirthPlace,Hagen-Haspe,NaN,NaN,NaN,Haspe,Hagen,NaN,NaN,NaN,NaN,NaN
48,val_27,VictimBirthPlace,Hohenrode,NaN,NaN,NaN,NaN,Hohenrode,NaN,NaN,NaN,NaN,NaN


## Prompting configuration

Same prompt template, supported entities, example count, embedding model and
similarity threshold as `Qwen3.5-9B-best-from-optuna` in
`cross_val_evaluation.ipynb`.

In [4]:
prompt_qwen_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_qwen_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35

# Already trained on this exact train/val split (see modules/train_ner.py),
# so it can be reused as-is instead of retraining per fold.
ner_model_dir = "models/ner_bzk"

## Build the example-matching strategy

Same hybrid NER-pattern + embedding-similarity strategy as the cross-val
notebook, built once from the train split (no per-fold loop needed since
there is only one split here).

In [5]:
pattern_similarity = llm_parsers.NERPatternSimilarExamples(
    example_addresses=train_data['FullAddress'].reset_index(drop=True),
    example_labels=train_data,
    labels_to_include=supported_entities,
    num_examples=n_examples,
    model_dir=ner_model_dir
)
embedding_similarity = llm_parsers.SimilarExamples(
    embedding_model=embedding_model,
    example_addresses=train_data['FullAddress'].reset_index(drop=True),
    example_labels=train_data,
    labels_to_include=supported_entities,
    num_examples=n_examples,
    similarity_threshold=similarity_threshold,
    device=device
)
hybrid_similarity = llm_parsers.HybridSimilarExamples(
    pattern_strategy=pattern_similarity,
    embedding_strategy=embedding_similarity,
    num_examples=n_examples,
    pool_size=n_examples
)

Computing NER patterns for 771 training examples...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Remote model

`max_workers` and `chunk_size` (below) are kept low on purpose to stay under
the inference server's rate limits; `max_retries` lets the OpenAI client
retry transient/rate-limit errors with backoff before giving up.

In [6]:
model = llm_parsers.RemoteAddressParsingModel(
    model_name="qwen3-30b-a3b-instruct-2507",
    credentials_path="ai_api_credentials.json",
    example_strategy=hybrid_similarity,
    prompt=prompt_qwen_optuna_best,
    max_workers=10,
    extra_client_kwargs={"max_retries": 5}
)

## Run on the validation split, with resumable caching

Predictions are cached to disk per-address and reloaded on rerun, so if a
chunk fails (e.g. rate limit exceeded), progress up to that point is kept and
re-running the cell picks up where it left off instead of re-querying
everything.

In [7]:
preds_path = Path("experiments_data") / "remote_val2" / "qwen3-30b-a3b-instruct-2507"
preds_file = preds_path / "preds.json"
preds_file.parent.mkdir(parents=True, exist_ok=True)

def run_validation_split(model, val_data):
    addresses = val_data['FullAddress'].tolist()[:10]
    start_time = time.monotonic()
    preds = model.parse_addresses(addresses)
    deltatime = time.monotonic() - start_time
    rate_limit_sleep_time = model.time_spent_on_rate_limit_sleep
    deltatime_net = deltatime - rate_limit_sleep_time
    print(f"Total time: {format_time(deltatime)}, time spent on rate limit sleep: {format_time(rate_limit_sleep_time)}, net time: {format_time(deltatime_net)}")
    print(f"Expected rate: {len(addresses) / deltatime_net:.2f} addresses/s")
    time_per_batch = deltatime_net / (len(addresses)/10)
    print(f"Time per batch of 10 addresses: {time_per_batch:.2f}s")
    print(f"Expected time for 4 394 539 addresses:\n\t{format_time(deltatime_net * 4_394_539 / len(addresses))}")
    preds_df = pd.DataFrame(preds)
    return preds_df


preds_df = run_validation_split(model, val_data)

Total time: 0:01:01, time spent on rate limit sleep: 0:01:00, net time: 0:00:01
Expected rate: 10.97 addresses/s
Time per batch of 10 addresses: 0.91s
Expected time for 4 394 539 addresses:
	4 days, 15:14:23


## Results

In [8]:
required_metrics = pd.Series(
    compare_preds(preds_df, val_data, target_columns=REQUIRED_ENTITIES),
    name="Country/City/Street/House"
)
display(required_metrics)

AssertionError: Length mismatch between preds and labels

In [ ]:
specific_metrics = OrderedDict()
for entity in supported_entities:
    specific_metrics[entity] = compare_preds(preds_df, val_data, target_columns=[entity])
display(pd.DataFrame(specific_metrics))

,HouseNumber,StreetName,Neighborhood,City,Country
accuracy,0.967105,0.973684,0.921053,0.921053,0.960526
precision,0.918033,0.937500,0.647059,0.925170,0.872340
recall,0.918033,0.967742,0.611111,0.931507,0.976190
f1,0.918033,0.952381,0.628571,0.928328,0.921348
accuracy_with_tol_1,0.967105,0.973684,0.921053,0.921053,0.960526
accuracy_with_tol_2,0.980263,0.973684,0.934211,0.921053,0.967105
accuracy_with_tol_3,0.993421,0.986842,0.934211,0.934211,0.973684
accuracy_with_tol_4,1.000000,0.986842,0.934211,0.934211,0.973684
average_levenshtein,0.092105,0.269737,0.677632,0.703947,0.315789
average_similarity,0.979308,0.980263,0.925282,0.938763,0.965643


### Results per BZK field

In [ ]:
BZK_ADDRESS_FIELDS = [
    'ApplicantCurrentAddress',
    'VictimBirthPlace',
    'VictimCurrentAddress',
    'ApplicantBirthPlace',
    'VictimDeathPlace'
]

field_metrics = OrderedDict()
for field in BZK_ADDRESS_FIELDS:
    mask = val_data['field'] == field
    field_metrics[field] = compare_preds(preds_df[mask], val_data[mask], target_columns=REQUIRED_ENTITIES)
display(pd.DataFrame(field_metrics))

,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
accuracy,0.906863,1.0,0.947368,0.986111,0.958333
precision,0.889571,1.0,0.928571,0.956522,0.800000
recall,0.917722,1.0,0.945455,0.970588,1.000000
f1,0.903427,1.0,0.936937,0.963504,0.888889
accuracy_with_tol_1,0.906863,1.0,0.947368,0.986111,0.958333
accuracy_with_tol_2,0.916667,1.0,0.947368,0.990741,0.958333
accuracy_with_tol_3,0.950980,1.0,0.947368,0.990741,0.958333
accuracy_with_tol_4,0.950980,1.0,0.960526,0.990741,0.958333
average_levenshtein,0.696078,0.0,0.486842,0.097222,0.416667
average_similarity,0.930857,1.0,0.954302,0.990291,0.958333
